In [30]:
# # this needs to run only once to load the model into memory
import easyocr
reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [46]:
#Initial Test
result = reader.readtext("Images/Test_2_Zoom_Out.png", detail = 0)# detail = 0 will not show you the probabilities

print(result)

['SANTA CLARA UNIVERSITY', 'Student Outcomes Self Assessment', 'CSEN 178, Jahangiri; Spring', '2026', 'Yqun(Lsdonsensbeenteconed']


/Users/timothyle/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [32]:
# Dataset Setup — IIIT5K-Words
# Download from: https://cvit.iiit.ac.in/research/projects/cvit-projects/the-iiit-5k-word-dataset
# Extract so you have: IIIT5K/train/ and IIIT5K/test/ folders of .png word images

import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class IIIT5KDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.paths = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith('.png') or f.endswith('.jpg')
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        # Return (image, 0) — label is unused, we just need the image tensor
        return img, 0

transform = transforms.Compose([
    transforms.Resize((32, 128)),   # consistent size for word images
    transforms.ToTensor(),          # converts to [0,1] float tensor
])

train_dataset = IIIT5KDataset('IIIT5KDownload/IIIT5K-Word_V3.0/IIIT5K/train', transform=transform)
dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Loaded {len(train_dataset)} training images")

Loaded 2000 training images


In [33]:
# Preprocessing that focuses on superesolution using a CNN  
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextEnhanceCNN(nn.Module):
    def __init__(self,scale_factor = 2):
        super().__init__()
        self.scale_factor = scale_factor# first addition of scale, will be used in other cells as well


        #Layer 1: extract basic features of the image
        # 1 = input channel(grayscale), 32 = number of features we learn, kernel_size = 3; 3x3x3 filter, padding = 1 so output stays the same size as the input
        #could change the number of featuers we learn to 8 or 16 or even 96 (AlexNet) can change later
        self.conv1 = nn.Conv2d(1, 32, kernel_size = 3, padding = 1)
        #choosing ReLU as activation function because it is simple and was used in the class for example
        self.relu1 = nn.ReLU()

        #Layer 2: learn more details

        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
        self.relu2 = nn.ReLU()

        #Layer 3: feature refinement

        self.conv3 = nn.Conv2d(64, 32, kernel_size = 3, padding = 1)
        self.relu3 =  nn.ReLU()

        #Output layer: bring back to a singular image
        #squished back from 32 to 1
        self.upsample = nn.ConvTranspose2d(
            32, 1,
            kernel_size=scale_factor * 2,   # kernel larger than stride avoids checkerboard artifacts
            stride=scale_factor,
            padding=scale_factor // 2
        )

    def forward(self, x):
        #switched from passing in x and updating x to instead updating out
        #this means that we don't need to rewrite the entire image and instead we just look for the errors
        out = self.relu1(self.conv1(x))
        out = self.relu2(self.conv2(out))
        out = self.relu3(self.conv3(out))
        out = torch.clamp(self.upsample(out), 0, 1)

        #out = self.sigmoid(self.conv_output(x))
        return out
    


In [34]:
#Training Model with Adam, Also replaced train to use downsampled input
'''
GOAL
 Training for super-resolution:
 Input  = image shrunk by scale_factor (simulates low-res)
 Target = original full-res image
 The model learns to reconstruct the detail that was lost
'''
def train(model, dataloader, epochs=10, lr=0.001):
    # Adam instead of SGD — converges much faster for CNNs
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()
    scale = model.scale_factor


    for epoch in range(epochs):
        total_loss = 0
        for clean_images, _ in dataloader:
            clean_images = clean_images.mean(dim=1, keepdim=True)

            # Simulate low-res: downsample then upsample back to input size
            # This gives the model a blurry/blocky version as input
            _, _, H, W = clean_images.shape
            low_res = F.interpolate(clean_images, size=(H // scale, W // scale), mode='bilinear', align_corners=False)
 
            #Now trying 
            output = model(low_res)
            target = F.interpolate(clean_images, size=output.shape[2:], mode='bilinear', align_corners=False)


            loss = loss_fn(output, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs} Loss: {avg_loss:.5f}")

    torch.save(model.state_dict(), "text_enhance_cnn.pth")
    print("Model saved to text_enhance_cnn.pth")

In [35]:
#NEW PREPROCESS; added clahe to improve the contrast of images
def preprocess(image_array, model):
    import numpy as np
    import cv2

    TILE_H, TILE_W = 32, 128  # must match CNN training size
    scale = model.scale_factor


    # Convert to grayscale
    if len(image_array.shape) == 3:
        gray = cv2.cvtColor(image_array, cv2.COLOR_BGR2GRAY)
    else:
        gray = image_array.copy()

    # CLAHE: fixes uneven lighting and low contrast
    # clipLimit controls how aggressively contrast is boosted — 2.0 is a safe default
    # tileGridSize divides the image into an 8x8 grid and equalizes each region independently
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Unsharp mask: sharpens edges by subtracting a blurred version from the original
    # The result is that edges (like text strokes) become more defined
    blurred = cv2.GaussianBlur(gray, (0, 0), 3)
    gray = cv2.addWeighted(gray, 1.5, blurred, -0.5, 0)


    orig_h, orig_w = gray.shape

    # Pad image so dimensions divide evenly into tiles
    pad_h = (TILE_H - orig_h % TILE_H) % TILE_H
    pad_w = (TILE_W - orig_w % TILE_W) % TILE_W
    padded = np.pad(gray, ((0, pad_h), (0, pad_w)), mode='reflect')
    padded_h, padded_w = padded.shape

    enhanced_padded = np.zeros((padded_h * scale, padded_w * scale), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for row in range(0, padded_h, TILE_H):
            for col in range(0, padded_w, TILE_W):
                tile = padded[row:row+TILE_H, col:col+TILE_W]

                # Normalize and convert to tensor: shape (1, 1, 32, 128)
                tensor = torch.from_numpy(tile).float() / 255.0
                tensor = tensor.unsqueeze(0).unsqueeze(0)

                # Run tile through CNN
                output = model(tensor)

                out_tile = output.squeeze().numpy()

                r0, c0 = row * scale, col * scale

                # Store result back
                enhanced_padded[r0:r0+out_tile.shape[0], c0:c0+out_tile.shape[1]] = out_tile


    # Crop back to original size, unnormalize to [0, 255]
    enhanced_np = (enhanced_padded[:orig_h * scale, :orig_w * scale] * 255).astype(np.uint8)

    # EasyOCR expects a 3-channel image
    return cv2.cvtColor(enhanced_np, cv2.COLOR_GRAY2BGR)

In [36]:
model = TextEnhanceCNN(scale_factor = 2)
train(model, dataloader, epochs=10)

Epoch 1/10 Loss: 0.02384
Epoch 2/10 Loss: 0.00258
Epoch 3/10 Loss: 0.00172
Epoch 4/10 Loss: 0.00152
Epoch 5/10 Loss: 0.00143
Epoch 6/10 Loss: 0.00133
Epoch 7/10 Loss: 0.00129
Epoch 8/10 Loss: 0.00125
Epoch 9/10 Loss: 0.00121
Epoch 10/10 Loss: 0.00119
Model saved to text_enhance_cnn.pth


In [45]:
#New test with ADAM optimzer
import numpy as np
import cv2

model = TextEnhanceCNN(scale_factor = 2)
model.load_state_dict(torch.load("text_enhance_cnn.pth"))
model.eval()
print("Loaded saved model")

raw_image = cv2.imread("Images/Test_2_Zoom_Out.png")
enhanced_image = preprocess(raw_image, model)

raw_result = reader.readtext(raw_image, detail=0)
enhanced_result = reader.readtext(enhanced_image, detail=0)

print("Raw OCR:     ", raw_result)
print("Enhanced OCR:", enhanced_result)

Loaded saved model
Raw OCR:      ['SANTA CLARA UNIVERSITY', 'Student Outcomes Self Assessment', 'CSEN 178, Jahangiri; Spring', '2026', 'Yqun(Lsdonsensbeenteconed']
Enhanced OCR: ['SANTA CLARA UNIVERSITY', 'Student Outcomes Self Assessment', 'CSEN 178, Jahangiri, Spring', '2026', 'Yaur response has been recoided.']


In [54]:
#Analytics
import time
import numpy as np

def run_analytics(image_path, model, reader):

    # Raw OCR timing
    raw_image = cv2.imread(image_path)
    H, W = raw_image.shape[:2]

    
    t0 = time.perf_counter()
    raw_result = reader.readtext(raw_image, detail=0)
    raw_time = time.perf_counter() - t0

    #Image enhancement timing
    t1 = time.perf_counter()
    enhanced_image = preprocess(raw_image, model)
    preprocess_time = time.perf_counter() - t1

    #Enhanced OCR timing
    t2 = time.perf_counter()
    enhanced_result = reader.readtext(enhanced_image, detail=0)
    enhanced_ocr_time = time.perf_counter() - t2

    #Total enhanced timing
    total_enhanced_time = preprocess_time + enhanced_ocr_time
    scale = model.scale_factor

    raw_pixels = H * W
    enhanced_pixels = (H * scale) * (W * scale)
    raw_throughput = raw_pixels/raw_time
    enhanced_throughput = enhanced_pixels / enhanced_ocr_time

    return {
        "image": image_path,
        "raw_time": raw_time,
        "preprocess_time": preprocess_time,
        "enhanced_ocr_time": enhanced_ocr_time,
        "total_enhanced_time": total_enhanced_time,
        "raw_pixels": raw_pixels,
        "enhanced_pixels": enhanced_pixels,
        "raw_throughput_mps": raw_throughput / 1e6,
        "enh_throughput_mps": enhanced_throughput / 1e6,
    }

results = []
for img_path in["Images/Test_2_Zoom_Out.png"]:
    results.append(run_analytics(img_path, model, reader))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/timothyle/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/timothyle/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/timothyle/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  File "

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import